In [2]:
from pathlib import Path
import sys

# Add the project root directory to sys.path
sys.path.append(str(Path().resolve().parent))
from src.constants.paths import SECRET_PATH
from src.processing.pde_ple import PDE, es
dic = PDE.s3.download(
    path="s3://bkt-pud-uc/uc202-rex/dictionary1.csv",
    local_file="/opt/app-root/src/uc202-ipn-rex/dictionary1.csv",
)


import csv

csv.field_size_limit(1000000)  # Set a higher limit, e.g., 1,000,000


def csv_to_string(file_path):
    with open(file_path, mode="r", encoding="utf-8") as file:
        csv_reader = csv.reader(file)
        # Read each row and join them into a single string
        csv_string = "\n".join([", ".join(row) for row in csv_reader])
    return csv_string


/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/elasticsearch/_sync/client/__init__.py:397: SecurityWarning: Connecting to 'https://noeyyalp.noe.edf.fr:29203' using TLS with verify_certs=False is insecure
  _transport = transport_class(


In [10]:
import re

# Exemple de texte brut
text = csv_to_string("/opt/app-root/src/uc202-ipn-rex/dictionary1.csv")
# Regex pour découper le texte
pattern = r'(Le sigle|Le code|Le trigramme|Le multigramme)\s*"([^"]+)"\s*signifie\s*(.*?)(?=(Le sigle|Le code|Le trigramme|Le multigramme|$))'

matches = re.finditer(pattern, text, re.DOTALL)

parsed_entries = []
for match in matches:
    sigle = match.group(2).strip()
    definition = match.group(3).strip().rstrip(".")
    parsed_entries.append(
        {
            "sigle": sigle,
            "definition": definition,
            "full_text": f'Le sigle "{sigle}" signifie {definition}.',
        }
    )


In [4]:
from src.processing.pde_ple import PDE, es
def get_all_sigles():
    """Fetch all sigles from the dictionary index"""
    query_body = {"size": 10000, "_source": ["sigle"], "query": {"match_all": {}}}

    response = es.search(index="lexique_nucleaire", body=query_body)
    sigles = [hit["_source"]["sigle"] for hit in response["hits"]["hits"]]
    return sigles

sigles=get_all_sigles()


/tmp/ipykernel_45523/369218566.py:6: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  response = es.search(index="lexique_nucleaire", body=query_body)
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [12]:
for i in parsed_entries:
    if i['sigle']=="F RRA 2":
        print(i)

{'sigle': 'F RRA 2', 'definition': 'Procédure de conduite. Mise hors service du circuit RRA. [ Cours FRAMATOME PWR/ETA/001 1300 B : p 17 ]', 'full_text': 'Le sigle "F RRA 2" signifie Procédure de conduite. Mise hors service du circuit RRA. [ Cours FRAMATOME PWR/ETA/001 1300 B : p 17 ].'}


In [11]:
import kitten

/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/kitten/loader.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-large")

for entry in parsed_entries:
    vector = model.encode(entry["definition"]).tolist()
    entry["embedding"] = vector
    es.index(index="lexique_nucleaire", body=entry)


[{'sigle': 'A',
  'definition': 'Alimentation en eau (des GV)',
  'full_text': 'Le sigle "A" signifie Alimentation en eau (des GV).'},
 {'sigle': 'A',
  'definition': 'classe de feu : matériaux solides',
  'full_text': 'Le sigle "A" signifie classe de feu : matériaux solides.'},
 {'sigle': 'A',
  'definition': 'message RTE : arrêter la baisse',
  'full_text': 'Le sigle "A" signifie message RTE : arrêter la baisse.'},
 {'sigle': 'A',
  'definition': "point d'Arrêt",
  'full_text': 'Le sigle "A" signifie point d\'Arrêt.'},
 {'sigle': 'a',
  'definition': "rayonnement alpha : noyau d'helium",
  'full_text': 'Le sigle "a" signifie rayonnement alpha : noyau d\'helium.'},
 {'sigle': 'A',
  'definition': 'travailleur DATR de catégorie A : max de 20mSv/12 mois',
  'full_text': 'Le sigle "A" signifie travailleur DATR de catégorie A : max de 20mSv/12 mois.'},
 {'sigle': 'A',
  'definition': 'audit et provient de Ensemble de SE',
  'full_text': 'Le sigle "A" signifie audit et provient de Ensemble